In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# CONFIGURACIÓN (lo que normalmente se edita)
# =============================================================================

MZ = 91.1876          # masa del bosón Z en GeV (PDG)
CHUNKSIZE = 200_000   # filas por bloque al leer los CSV


ETIQUETA = "estricto"

# ---------- Rutas ----------
CARPETA = "D:/proyecto_cern/"
SALIDA = CARPETA + "cutflow 70-180/"

# Los 4 CSV de datos reales (se procesan y se unen como un solo conjunto)
DATOS_REALES = [
    CARPETA + "Run2012B_DoubleElectron_Data_Aplanada.csv",
    CARPETA + "Run2012B_DoubleMuParked_Data_Aplanada.csv",
    CARPETA + "Run2012C_DoubleElectron_Data_Aplanada.csv",
    CARPETA + "Run2012C_DoubleMuParked_Data_Aplanada.csv",
]

# CSV de 4 leptones de las simulaciones 
RUTAS = {
    "SMHiggsToZZTo4L": CARPETA + "SMHiggs_4lep.csv",
    "ZZTo4mu":         CARPETA + "ZZTo4mu_4lep.csv",
    "ZZTo4e":          CARPETA + "ZZTo4e_4lep.csv",
    "ZZTo2e2mu":       CARPETA + "ZZTo2e2mu_4lep.csv",
}
MOSTRAR = False   # True = además de guardar el PNG, lo muestra en pantalla

# ---------- Luminosidad y muestras simuladas ----------
LUMI = 11580.0   # luminosidad integrada en pb^-1 (= 11.58 fb^-1)

# xsec : sección eficaz en pb
# nevt : eventos GENERADOS en la simulación (no las filas del CSV)
# k    : K-factor (corrección a órdenes superiores; 1.386 para el fondo ZZ)
# tipo : "senal" (Higgs) o "fondo" (ZZ); define cómo se dibuja en la gráfica
MUESTRAS = {
    "SMHiggsToZZTo4L": {"xsec": 0.0065, "nevt": 299973.0,  "k": 1.0,   "tipo": "senal"},
    "ZZTo4mu":         {"xsec": 0.077,  "nevt": 1499064.0, "k": 1.386, "tipo": "fondo"},
    "ZZTo4e":          {"xsec": 0.077,  "nevt": 1499093.0, "k": 1.386, "tipo": "fondo"},
    "ZZTo2e2mu":       {"xsec": 0.18,   "nevt": 1497445.0, "k": 1.386, "tipo": "fondo"},
}

# =============================================================================
# VALORES DE LOS CORTES (selección ESTRICTA)
# =============================================================================
CUTS = {
    # --- Cinemática mínima de cada leptón ---
    "pt_e_min": 7,        # electrones: pT > 7 GeV
    "pt_mu_min": 5,       # muones:     pT > 5 GeV
    "eta_e_max": 2.5,     # electrones: |eta| < 2.5 (cobertura del detector)
    "eta_mu_max": 2.4,    # muones:     |eta| < 2.4 (cobertura de las cámaras de muones)

    # --- Compatibilidad con el vértice primario (descarta leptones "no prompt") ---
    "dxy_max": 0.5,       # |dxy| < 0.5 cm
    "dz_max": 1.0,        # |dz|  < 1 cm
    "sip_max": 4.0,       # SIP3D < 4
    "iso_max": 0.4,       # aislamiento relativo < 0.4

    # --- Leptones más duros del evento ---
    "pt1_min": 20,        # el leptón de mayor pT debe tener pT > 20 GeV
    "pt2_min": 10,        # el segundo de mayor pT debe tener pT > 10 GeV

    # --- Masas de los pares (bosones Z) ---
    # Z1 = el par más cercano a la masa del Z; Z2 = el otro par.
    "mz1_min": 40, "mz1_max": 120,
    "mz2_min": 12, "mz2_max": 120,
    "mll_min": 4,         # cualquier par de carga opuesta debe tener masa > 4 GeV
                          # (elimina resonancias de quarks pesados de baja masa)
    "m4l_min": 70,        # masa de los 4 leptones > 70 GeV
}

# Bordes de los bins del histograma: 36 bins de igual ancho entre 70 y 180 GeV
# (37 bordes), como en el artículo. Los eventos fuera de este rango no se dibujan.
BINS = np.linspace(70, 180, 37)

# Columnas que existen para cada leptón (sufijo 1..4 en el CSV)
COLS_LEP = ["PID", "Q", "pt", "eta", "phi", "mass", "dxy", "dxyErr", "dz", "dzErr",
            "pfRelIso03_all", "pfRelIso04_all"]
# Conjunto de columnas que se leen del CSV (las demás se ignoran para ahorrar memoria)
NECESARIAS = {f"{c}{i}" for c in COLS_LEP for i in range(1, 5)} | {"run", "luminosityBlock", "event"}
# Columnas que no pueden faltar para considerar "válido" un leptón (corte 1)
COLS_VALIDAS = [f"{v}{i}" for v in ["PID", "Q", "pt", "eta", "phi", "mass"] for i in range(1, 5)]


# =============================================================================
# FÍSICA
# =============================================================================

def p4(df, i):
    """
    Cuadrivector (E, px, py, pz) del leptón i para todos los eventos del bloque.

    Se pasa de las variables (pt, eta, phi, masa) a componentes cartesianas:
        px = pT cos(phi),  py = pT sin(phi),  pz = pT sinh(eta)
        E  = sqrt(px^2 + py^2 + pz^2 + m^2)
    Devuelve un arreglo de forma (n_eventos, 4).
    """
    pt = df[f"pt{i}"].to_numpy()
    eta = df[f"eta{i}"].to_numpy()
    phi = df[f"phi{i}"].to_numpy()
    m = df[f"mass{i}"].to_numpy()
    px, py, pz = pt * np.cos(phi), pt * np.sin(phi), pt * np.sinh(eta)
    E = np.sqrt(px**2 + py**2 + pz**2 + m**2)
    return np.stack([E, px, py, pz], axis=1)


def inv_mass(*vecs):
    """
    Masa invariante de un sistema de partículas: suma los cuadrivectores y calcula
        m^2 = E^2 - px^2 - py^2 - pz^2
    (el clip a 0 evita raíces de números negativos pequeños por redondeo).
    """
    s = sum(vecs)
    m2 = s[:, 0]**2 - s[:, 1]**2 - s[:, 2]**2 - s[:, 3]**2
    return np.sqrt(np.clip(m2, 0, None))


def sip3d(dxy, dxyErr, dz, dzErr):
    """
    Significancia del parámetro de impacto en 3D (SIP3D) = IP / sigma_IP.

    Es una APROXIMACION: se toma IP = sqrt(dxy^2 + dz^2) y su incertidumbre se
    obtiene por propagación de errores SIN correlaciones entre dxy y dz (el CSV
    no trae la matriz de covarianza completa). Un SIP3D grande indica que el
    leptón no viene del vértice primario (probablemente de un decaimiento de
    quarks pesados), por lo que se descarta.
    """
    dxy = np.asarray(dxy, float)
    dxyErr = np.asarray(dxyErr, float)
    dz = np.asarray(dz, float)
    dzErr = np.asarray(dzErr, float)
    ip = np.sqrt(dxy**2 + dz**2)
    sig = np.sqrt((dxy**2 * dxyErr**2 + dz**2 * dzErr**2) / np.clip(ip**2, 1e-12, None))
    return ip / np.clip(sig, 1e-12, None)


# =============================================================================
# SELECCIÓN (todos los cortes de una vez)
# =============================================================================

def seleccion_final(df, C, con_claves=False):
    """
    Aplica TODOS los cortes a un bloque (chunk) del CSV.

    Parámetros
    ----------
    df         : bloque del CSV (DataFrame).
    C          : diccionario con los valores de los cortes (CUTS).
    con_claves : True para datos reales; además del m4l devuelve una clave única
                 por evento (hash de run, luminosityBlock, event) para quitar
                 duplicados entre datasets más adelante.

    Devuelve
    --------
    (m4l, claves) de los eventos que sobreviven a todos los cortes
    (claves es None si con_claves=False).

    Se usa una máscara booleana 'ok' que empieza en True para todos los eventos
    y se va "apagando" (con &=) cada vez que un evento falla un corte.
    """
    claves = None
    if con_claves:
        claves = pd.util.hash_pandas_object(
            df[["run", "luminosityBlock", "event"]], index=False).to_numpy()

    # ---------- Corte 1: 4 leptones válidos ----------
    # El valor -999 es relleno ("no hay dato"); se convierte a NaN y se descartan
    # los eventos a los que les falte cualquier dato básico de algún leptón.
    df = df.replace(-999, np.nan)
    valido = df[COLS_VALIDAS].notna().all(axis=1).to_numpy()
    df = df[valido]
    if claves is not None:
        claves = claves[valido]
    n = len(df)
    if n == 0:   # bloque sin eventos válidos: no hay nada más que hacer
        return np.array([], np.float32), (np.array([], np.uint64) if con_claves else None)

    v = [p4(df, i) for i in range(1, 5)]            # cuadrivectores de los 4 leptones
    m4l = inv_mass(*v)                              # masa invariante de los 4 leptones
    pid = [df[f"PID{i}"].abs().to_numpy() for i in range(1, 5)]   # |PID|: 11 = e, 13 = mu
    q = [df[f"Q{i}"].to_numpy() for i in range(1, 5)]             # cargas

    ok = np.ones(n, bool)

    # ---------- Cortes 2-6: por leptón (se exigen a LOS 4 leptones) ----------
    for i in range(1, 5):
        es_e = pid[i - 1] == 11      # ¿el leptón i es electrón?
        es_mu = pid[i - 1] == 13     # ¿el leptón i es muón?
        pt = df[f"pt{i}"].to_numpy()
        eta = np.abs(df[f"eta{i}"].to_numpy())
        dxy = df[f"dxy{i}"].to_numpy(float)
        dz = df[f"dz{i}"].to_numpy(float)

        # pT mínimo (distinto para electrones y muones)
        ok &= (es_e & (pt > C["pt_e_min"])) | (es_mu & (pt > C["pt_mu_min"]))
        # |eta| máximo (distinto para electrones y muones)
        ok &= (es_e & (eta < C["eta_e_max"])) | (es_mu & (eta < C["eta_mu_max"]))
        # parámetros de impacto transversal y longitudinal respecto al vértice
        ok &= (np.abs(dxy) < C["dxy_max"]) & (np.abs(dz) < C["dz_max"])

        # SIP3D (aproximado, ver sip3d). Si no se puede calcular (NaN), el leptón
        # NO se descarta por este corte.
        sip = sip3d(dxy, df[f"dxyErr{i}"], dz, df[f"dzErr{i}"])
        ok &= (sip < C["sip_max"]) | np.isnan(sip)

        # Aislamiento relativo: los muones usan el cono de radio 0.4 (pfRelIso04);
        # los electrones no tienen esa columna (NaN), así que se usa el cono 0.3
        # (pfRelIso03). fillna rellena solo donde falta la de 0.4.
        iso = df[f"pfRelIso04_all{i}"].fillna(df[f"pfRelIso03_all{i}"]).to_numpy()
        ok &= iso < C["iso_max"]

    # ---------- Corte 7: carga total cero ----------
    # Los 4 leptones vienen de dos Z neutros, así que las cargas deben sumar 0.
    ok &= (q[0] + q[1] + q[2] + q[3]) == 0

    # ---------- Corte 8: pT de los dos leptones más duros ----------
    # Se ordenan los 4 pT de mayor a menor y se exige pT1 > 20 y pT2 > 10 GeV
    # (para asegurar que el evento pudo ser disparado por el trigger).
    pts = np.sort(np.stack([df[f"pt{i}"].to_numpy() for i in range(1, 5)], axis=1), axis=1)[:, ::-1]
    ok &= (pts[:, 0] > C["pt1_min"]) & (pts[:, 1] > C["pt2_min"])

    # ---------- mll mínima (se calcula ahora, se APLICA en el corte 12) ----------
    # Para cada evento: la menor masa de entre TODOS los pares de carga opuesta
    # (6 pares posibles entre 4 leptones, de cualquier sabor).
    mll_min = np.full(n, np.inf)
    for a in range(4):
        for b in range(a + 1, 4):
            os_ = (q[a] + q[b]) == 0                 # ¿par de carga opuesta?
            mab = inv_mass(v[a], v[b])
            mll_min = np.where(os_, np.minimum(mll_min, mab), mll_min)

    # ---------- Corte 9: emparejamiento en dos Z ----------
    # Los 4 leptones se pueden agrupar en 2 pares de 3 formas distintas:
    combos = [((0, 1), (2, 3)), ((0, 2), (1, 3)), ((0, 3), (1, 2))]

    def par_valido(a, b):
        """Par válido = mismo sabor (e-e o mu-mu) y carga opuesta."""
        return (pid[a] == pid[b]) & (q[a] + q[b] == 0)

    # Entre las combinaciones en que AMBOS pares son válidos, se elige la que
    # tenga el par más cercano a la masa del Z (ese par es Z1 y el otro Z2).
    mejor_score = np.full(n, np.inf)   # |mZ1 - MZ| de la mejor combinación hasta ahora
    mZ1 = np.full(n, np.nan)
    mZ2 = np.full(n, np.nan)
    for (a, b), (c, d) in combos:
        valido_c = par_valido(a, b) & par_valido(c, d)
        mab = inv_mass(v[a], v[b])
        mcd = inv_mass(v[c], v[d])
        ab_es_z1 = np.abs(mab - MZ) < np.abs(mcd - MZ)   # ¿(a,b) está más cerca de MZ?
        z1 = np.where(ab_es_z1, mab, mcd)
        z2 = np.where(ab_es_z1, mcd, mab)
        score = np.abs(z1 - MZ)
        mejor = valido_c & (score < mejor_score)
        mejor_score = np.where(mejor, score, mejor_score)
        mZ1 = np.where(mejor, z1, mZ1)
        mZ2 = np.where(mejor, z2, mZ2)

    # Si ninguna combinación fue válida, mZ1 queda NaN y el evento se descarta.
    ok &= np.isfinite(mZ1)
    # ---------- Cortes 10 y 11: ventanas de masa de Z1 y Z2 ----------
    ok &= (mZ1 > C["mz1_min"]) & (mZ1 < C["mz1_max"])
    ok &= (mZ2 > C["mz2_min"]) & (mZ2 < C["mz2_max"])
    # ---------- Corte 12: mll mínima ----------
    ok &= mll_min > C["mll_min"]
    # ---------- Corte 13: m4l mínima ----------
    ok &= m4l > C["m4l_min"]

    return m4l[ok].astype(np.float32), (None if claves is None else claves[ok])


def procesar(rutas, nombre, con_claves):
    """
    Lee uno o varios CSV por bloques y devuelve el arreglo de m4l de los eventos
    que pasan todos los cortes.

    Con con_claves=True (datos reales) se quitan los eventos duplicados entre
    archivos usando la clave (run, luminosityBlock, event); de cada evento
    repetido se conserva la primera aparición.
    """
    m4l_lista, claves_lista = [], []
    leidos = 0
    for ruta in rutas:
        for bloque in pd.read_csv(ruta, chunksize=CHUNKSIZE,
                                  usecols=lambda c: c in NECESARIAS):
            m, cl = seleccion_final(bloque, CUTS, con_claves)
            m4l_lista.append(m)
            if cl is not None:
                claves_lista.append(cl)
            leidos += len(bloque)
            print(f"  {nombre}: {leidos} filas leidas", end="\r")
    print()

    m4l = np.concatenate(m4l_lista) if m4l_lista else np.array([], np.float32)
    if con_claves and claves_lista:
        # Eliminación de duplicados: np.unique devuelve el índice de la primera
        # aparición de cada clave; con esos índices se filtra el arreglo de m4l.
        cl = np.concatenate(claves_lista)
        _, idx = np.unique(cl, return_index=True)
        m4l = m4l[idx]
    return m4l


def peso_por_evento(nombre):
    """Peso de un evento simulado: w = L * sigma * k / N_generados."""
    m = MUESTRAS[nombre]
    return LUMI * m["xsec"] * m["k"] / m["nevt"]


# =============================================================================
# GRÁFICA FINAL
# =============================================================================

def grafica_final(m4l_sim, m4l_datos, archivo, bins=BINS):
    """
    Estilo CMS Open Data: datos reales como puntos con incertidumbre estadística
    (raíz de N) sobre el fondo ZZ (azul) y la señal del Higgs (línea roja).

    Parámetros
    ----------
    m4l_sim   : {nombre de muestra: arreglo de m4l tras los cortes} del MC.
    m4l_datos : arreglo de m4l de los datos reales tras los cortes.
    archivo   : ruta del PNG de salida.
    bins      : bordes de los bins.
    """
    # --- simulaciones ponderadas por luminosidad y sección eficaz ---
    # h_zz: suma de los tres canales ZZ (fondo); h_h: Higgs (señal).
    h_zz = np.zeros(len(bins) - 1)
    h_h = np.zeros(len(bins) - 1)
    for nombre, m4l in m4l_sim.items():
        h, _ = np.histogram(m4l, bins=bins, weights=np.full(len(m4l), peso_por_evento(nombre)))
        if MUESTRAS[nombre]["tipo"] == "senal":
            h_h += h
        else:
            h_zz += h
    h_tot = h_zz + h_h     # fondo + señal apilados (línea roja)

    # --- datos reales: conteo por bin y error de Poisson (sqrt(N)) ---
    h_dat, _ = np.histogram(m4l_datos, bins=bins)
    err = np.sqrt(h_dat)
    centros = 0.5 * (bins[:-1] + bins[1:])
    con_datos = h_dat > 0          # solo se dibujan puntos en bins con eventos

    print(f"\nGrafica final ({bins[0]:.0f}-{bins[-1]:.0f} GeV): datos = {int(h_dat.sum())}, "
          f"ZZ esperado = {h_zz.sum():.2f}, Higgs esperado = {h_h.sum():.2f}")

    fig, ax = plt.subplots(figsize=(8, 7))
    # Primero la línea roja (ZZ + Higgs), luego el ZZ relleno encima, y al final los datos.
    ax.stairs(h_tot, bins, color="red", linewidth=2, label="m$_H$ = 125 GeV")
    ax.stairs(h_zz, bins, fill=True, facecolor="#9ecae1", edgecolor="black",
              linewidth=1.5, label="ZZ")
    ax.errorbar(centros[con_datos], h_dat[con_datos], yerr=err[con_datos],
                fmt="o", color="black", markersize=5, elinewidth=1.3, capsize=0,
                label="Data", zorder=5)

    # El eje Y llega a 25 % por encima del valor máximo (datos + error, o MC).
    tope = max((h_dat + err).max(), h_tot.max())
    ax.set_ylim(0, (tope if tope > 0 else 1) * 1.25)
    ax.set_xlim(bins[0], bins[-1])
    ax.set_xlabel("Mass 4 leptons / GeV", fontsize=12)
    ax.set_ylabel("N_Events", fontsize=12)
    ax.text(0.02, 1.02, "CMS Open Data", transform=ax.transAxes, fontsize=12, fontweight="bold")
    ax.text(0.98, 1.02, r"$\sqrt{s}$ = 8 TeV, L$_{int}$ = 11.6 fb$^{-1}$",
            transform=ax.transAxes, fontsize=11, ha="right")

    # leyenda en el orden del artículo: Data, ZZ, Higgs
    handles, labels = ax.get_legend_handles_labels()
    d = dict(zip(labels, handles))
    orden = ["Data", "ZZ", "m$_H$ = 125 GeV"]
    ax.legend([d[k] for k in orden], orden, frameon=False,
              loc="upper right", bbox_to_anchor=(0.97, 0.95))

    fig.tight_layout()
    fig.savefig(archivo, dpi=150)
    if MOSTRAR:
        plt.show()
    plt.close(fig)


# =============================================================================
# PROGRAMA PRINCIPAL
# =============================================================================

if __name__ == "__main__":
    os.makedirs(SALIDA, exist_ok=True)

    # 1) Simulaciones: cada muestra se procesa por separado (sin quitar duplicados)
    m4l_sim = {}
    for nombre, ruta in RUTAS.items():
        print(f"Procesando {nombre}...")
        m4l_sim[nombre] = procesar([ruta], nombre, con_claves=False)

    # 2) Datos reales: los 4 CSV se procesan juntos, quitando eventos repetidos
    print("Procesando datos reales (4 datasets unidos)...")
    m4l_datos = procesar(DATOS_REALES, "Datos reales", con_claves=True)

    # 3) Gráfica final
    grafica_final(m4l_sim, m4l_datos, SALIDA + f"grafica_final_{ETIQUETA}.png")